# TikTok EDA — Understanding Our Data

This notebook explores the TikTok data we scraped. The goal is to answer:
- What does a typical video look like in our dataset?
- What makes a video perform well?
- How do users interact (comments, replies)?
- Who are the creators in our data?

We use the **silver layer** — data already cleaned, deduplicated, and enriched with derived fields like `engagement_rate`.

> Each section has a plot followed by a plain-English explanation of what it means and why it matters for building a recommendation system.

In [ ]:
import glob
import os
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

SILVER_ROOT = '../extracts/silver'
runs = sorted(
    [r for r in glob.glob(os.path.join(SILVER_ROOT, '*')) if os.path.isdir(r)],
    reverse=True
)
latest = runs[0]
print('Using run:', latest)

In [ ]:
videos   = pd.read_json(os.path.join(latest, 'videos.jsonl'),   lines=True)
comments = pd.read_json(os.path.join(latest, 'comments.jsonl'), lines=True)
authors  = pd.read_csv(os.path.join(latest, 'authors.csv'))

# Cast IDs to string
videos['video_id']     = videos['video_id'].astype(str)
comments['video_id']   = comments['video_id'].astype(str)
comments['comment_id'] = comments['comment_id'].astype(str)
authors['author_id']   = authors['author_id'].astype(str)

print(f'videos:   {len(videos):,} rows')
print(f'comments: {len(comments):,} rows')
print(f'authors:  {len(authors):,} rows')

---
## Part 1 — Videos

These plots answer: *What kind of videos are in our dataset, and which ones perform well?*

### Plot 1 — Engagement Rate Distribution

**Engagement rate** = (likes + comments + shares) / plays

It answers: out of everyone who *watched* the video, what % actually interacted with it? A high engagement rate means the content resonated strongly.

In [ ]:
er = videos['engagement_rate'].dropna()
er_clipped = er[er < er.quantile(0.99)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(er_clipped, bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(er.median(), color='red', linestyle='--', label=f'Median: {er.median():.3f}')
axes[0].set_title('Engagement Rate Distribution')
axes[0].set_xlabel('Engagement Rate  (interactions / plays)')
axes[0].set_ylabel('Number of Videos')
axes[0].legend()

axes[1].boxplot(er_clipped, vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].set_title('Engagement Rate Spread')
axes[1].set_ylabel('Engagement Rate')
axes[1].set_xticks([])

plt.suptitle('How engaging are the videos in our dataset?', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
print(er.describe().round(4))

**What this tells us:**
- Most videos have a low engagement rate — totally normal on TikTok. Most people scroll past without interacting.
- The distribution is **right-skewed**: a small number of videos get extremely high engagement.
- The red line (median) = the typical video in our dataset.
- For a recommendation system, `engagement_rate` is one of the most important signals — we want to recommend videos users are likely to interact with, not just watch.

### Plot 2 — Likes, Plays, Shares, Comments Distributions

These are the raw interaction counts. We look at all four together to understand what "normal" looks like for each metric.

In [ ]:
metrics = ['likes', 'plays', 'shares', 'comments_count']
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, col in zip(axes, metrics):
    data = videos[col].dropna()
    clipped = data[data < data.quantile(0.95)]
    ax.hist(clipped, bins=40, color='teal', edgecolor='white')
    median_val = data.median()
    ax.axvline(median_val, color='red', linestyle='--', linewidth=1.5, label=f'Median: {median_val:,.0f}')
    ax.set_title(col.replace('_', ' ').title())
    ax.set_xlabel('Count')
    ax.set_ylabel('Videos')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.tick_params(axis='x', rotation=30)
    ax.legend(fontsize=8)

plt.suptitle('Raw Interaction Counts per Video  (top 5% removed for readability)', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

**What this tells us:**
- All four metrics are heavily skewed right — most videos have low counts, a few go viral.
- `plays` is always much larger than `likes`, `shares`, and `comments` — most viewers are passive.
- Shares are the rarest interaction — when someone shares, it's a very strong quality signal.
- The gap between plays and likes is exactly what `engagement_rate` captures.

### Plot 3 — Top 20 Videos by Engagement Rate

Which specific videos in our dataset performed best? High engagement rate doesn't always mean high views — a small video with lots of interaction is very relevant.

In [ ]:
top20 = videos.nlargest(20, 'engagement_rate')[
    ['video_id', 'video_author_username', 'engagement_rate', 'plays', 'likes']
].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(14, 7))
bars = ax.barh(
    top20['video_author_username'] + '  (' + top20['plays'].map('{:,}'.format) + ' plays)',
    top20['engagement_rate'], color='coral'
)
ax.set_xlabel('Engagement Rate')
ax.set_title('Top 20 Videos by Engagement Rate')
ax.invert_yaxis()
for bar, val in zip(bars, top20['engagement_rate']):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()
print(top20[['video_author_username', 'engagement_rate', 'plays', 'likes']].to_string(index=False))

**What this tells us:**
- The bar = engagement rate. The label in parentheses = how many people watched it.
- Watch for videos with very high engagement but very few plays — niche but deeply resonant content.
- A recommender could use this list as "gold standard" examples of high-quality content to train on.

### Plot 4 — Video Duration Distribution

Does video length matter? Short-form content is TikTok's identity, but longer videos are increasingly common. This shows what we actually scraped.

In [ ]:
dur = videos['duration_sec'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(dur, bins=40, color='mediumpurple', edgecolor='white')
axes[0].axvline(dur.median(), color='red', linestyle='--', label=f'Median: {dur.median():.0f}s')
axes[0].set_title('Video Duration Distribution')
axes[0].set_xlabel('Duration (seconds)')
axes[0].set_ylabel('Number of Videos')
axes[0].legend()

bins   = [0, 15, 30, 60, 180, float('inf')]
labels = ['≤15s', '16–30s', '31–60s', '1–3 min', '>3 min']
dur_cat = pd.cut(dur, bins=bins, labels=labels)
counts  = dur_cat.value_counts().reindex(labels)
axes[1].bar(counts.index, counts.values, color='mediumpurple', edgecolor='white')
axes[1].set_title('Videos by Duration Category')
axes[1].set_xlabel('Duration Range')
axes[1].set_ylabel('Number of Videos')
for i, v in enumerate(counts.values):
    axes[1].text(i, v + 0.5, str(v), ha='center', fontsize=10)

plt.suptitle('How long are the videos in our dataset?', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
print(dur.describe().round(1))

**What this tells us:**
- The right chart groups videos into readable buckets — easier than reading the raw histogram.
- If most videos are ≤30s, our dataset reflects classic short-form TikTok.
- Duration is a useful recommendation feature — users who prefer short clips likely don't want 3-minute videos.

### Plot 5 — Does Duration Affect Engagement?

This scatter plot directly tests whether longer videos get more or less engagement.

In [ ]:
dur_er = videos[['duration_sec', 'engagement_rate']].dropna()
dur_er = dur_er[dur_er['engagement_rate'] < dur_er['engagement_rate'].quantile(0.99)]

fig, ax = plt.subplots(figsize=(12, 5))
ax.scatter(dur_er['duration_sec'], dur_er['engagement_rate'], alpha=0.3, s=15, color='steelblue')

z = np.polyfit(dur_er['duration_sec'], dur_er['engagement_rate'], 1)
p = np.poly1d(z)
x_line = np.linspace(dur_er['duration_sec'].min(), dur_er['duration_sec'].max(), 200)
ax.plot(x_line, p(x_line), color='red', linewidth=2, label='Trend')

ax.set_xlabel('Duration (seconds)')
ax.set_ylabel('Engagement Rate')
ax.set_title('Does Video Length Affect Engagement?')
ax.legend()
plt.tight_layout()
plt.show()

corr = dur_er['duration_sec'].corr(dur_er['engagement_rate'])
print(f'Correlation between duration and engagement rate: {corr:.3f}')

**What this tells us:**
- Each dot is one video. The red line is the overall trend.
- If the trend is **flat or negative**, duration doesn't help — shorter videos engage just as well.
- The correlation number confirms: close to 0 = no relationship, negative = shorter is better.
- Don't recommend long videos just because they have high play counts.

### Plot 6 — Top Hashtags

Hashtags are how TikTok categorizes content. They're crucial for a recommender — they tell us what topics and trends are in our dataset.

In [ ]:
all_tags = []
for tags in videos['hashtags'].dropna():
    if isinstance(tags, list):
        all_tags.extend([t.lower().strip('#') for t in tags if t])

if all_tags:
    tag_counts = Counter(all_tags)
    top_tags   = pd.DataFrame(tag_counts.most_common(30), columns=['hashtag', 'count'])

    fig, ax = plt.subplots(figsize=(14, 7))
    bars = ax.barh(top_tags['hashtag'], top_tags['count'], color='darkcyan')
    ax.set_xlabel('Number of Videos')
    ax.set_title('Top 30 Hashtags in Our Dataset')
    ax.invert_yaxis()
    for bar, val in zip(bars, top_tags['count']):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
                str(val), va='center', fontsize=8)
    plt.tight_layout()
    plt.show()

    print(f'Total unique hashtags: {len(tag_counts):,}')
    print(f'Videos with at least one hashtag: {videos["hashtags"].apply(lambda x: isinstance(x, list) and len(x) > 0).sum():,}')
else:
    print('No hashtag data found in this extract.')

**What this tells us:**
- The most common hashtags reveal the dominant topics in our dataset.
- Generic tags like `#fyp` or `#foryoupage` appear everywhere — they're noise for recommendation.
- Topic-specific hashtags (e.g. `#finance`, `#cooking`) are the ones that help us understand content categories.
- For a recommendation system, hashtags can match users with relevant topics (content-based filtering).

### Plot 7 — When Are Videos Posted?

This shows what hours of the day our scraped videos were published. Posting time is a weak but real signal.

In [ ]:
videos['created_at_dt'] = pd.to_datetime(videos['created_at'], utc=True, errors='coerce')
hour_counts = videos['created_at_dt'].dt.hour.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(hour_counts.index, hour_counts.values, color='steelblue', edgecolor='white')
ax.set_xticks(range(0, 24))
ax.set_xticklabels([f'{h:02d}:00' for h in range(24)], rotation=45)
ax.set_xlabel('Hour of Day (UTC)')
ax.set_ylabel('Number of Videos Posted')
ax.set_title('What Time of Day Are Videos Posted? (UTC)')
plt.tight_layout()
plt.show()

**What this tells us:**
- Peaks reveal when creators are most active.
- Timestamps are UTC — if most creators are in one region, you'll see a cluster around their local evening.
- Videos posted at peak hours reach more users but also compete with more content.

---
## Part 2 — Comments

These plots answer: *How do users talk about videos? What does text engagement look like?*

### Plot 8 — Comment Length Distribution

Short comments like "😂" or "lol" are very different from long thoughtful ones. Text length is a proxy for how much effort a user put in.

In [ ]:
tl        = comments['text_length'].dropna()
tl_clipped = tl[tl < tl.quantile(0.99)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(tl_clipped, bins=50, color='goldenrod', edgecolor='white')
axes[0].axvline(tl.median(), color='red', linestyle='--', label=f'Median: {tl.median():.0f} chars')
axes[0].set_title('Comment Length Distribution')
axes[0].set_xlabel('Number of Characters')
axes[0].set_ylabel('Number of Comments')
axes[0].legend()

buckets       = pd.cut(tl, bins=[0, 10, 30, 60, 100, float('inf')],
                       labels=['≤10', '11–30', '31–60', '61–100', '>100'])
bucket_counts = buckets.value_counts().sort_index()
axes[1].bar(bucket_counts.index, bucket_counts.values, color='goldenrod', edgecolor='white')
axes[1].set_title('Comments by Length Category')
axes[1].set_xlabel('Characters')
axes[1].set_ylabel('Number of Comments')
for i, v in enumerate(bucket_counts.values):
    axes[1].text(i, v + 0.5, str(v), ha='center', fontsize=9)

plt.suptitle('How long are comments in our dataset?', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(tl.describe().round(1))

short = comments[comments['text_length'] < 5]['text'].dropna().sample(
    min(5, len(comments[comments['text_length'] < 5])), random_state=42).tolist()
long_ = comments[comments['text_length'] > 100]['text'].dropna().sample(
    min(3, len(comments[comments['text_length'] > 100])), random_state=42).tolist()
print('\nExample short comments:', short)
print('\nExample long comments:', long_[:2])

**What this tells us:**
- Most TikTok comments are very short — emojis, single words, or short reactions.
- The sample prints real comments so you can sanity-check the data.
- For NLP/sentiment analysis, very short comments carry little signal. Long comments = deeper engagement.

### Plot 9 — Replies vs Top-level Comments

A reply means a user was engaged enough to respond to someone else. This tells us how much conversation is happening vs simple reactions.

In [ ]:
reply_counts = comments['is_reply'].value_counts()
labels       = ['Top-level' if not k else 'Reply' for k in reply_counts.index]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].pie(reply_counts, labels=labels, autopct='%1.1f%%',
            colors=['steelblue', 'coral'], startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[0].set_title('Top-level Comments vs Replies')

avg_len = comments.groupby('is_reply')['text_length'].mean()
axes[1].bar(['Top-level', 'Reply'], avg_len.values, color=['steelblue', 'coral'], edgecolor='white')
axes[1].set_title('Avg Comment Length: Top-level vs Reply')
axes[1].set_ylabel('Avg Characters')
for i, v in enumerate(avg_len.values):
    axes[1].text(i, v + 0.3, f'{v:.1f}', ha='center', fontsize=11)

plt.suptitle('Comment Structure in Our Dataset', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**What this tells us:**
- If replies are rare, our dataset is mostly first-reaction comments — less conversational depth.
- The right chart shows whether replies tend to be longer (more meaningful).
- Videos that trigger many replies are generating real conversations — a strong engagement signal.

### Plot 10 — Comments per Video

Are comments spread evenly across videos, or concentrated on a few viral ones? This tells us about the shape of user activity.

In [ ]:
comments_per_video = comments.groupby('video_id').size().reset_index(name='comment_count')
cpv = comments_per_video['comment_count']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(cpv[cpv < cpv.quantile(0.95)], bins=40, color='teal', edgecolor='white')
axes[0].axvline(cpv.median(), color='red', linestyle='--', label=f'Median: {cpv.median():.0f}')
axes[0].set_title('Comments per Video (top 5% removed)')
axes[0].set_xlabel('Number of Comments')
axes[0].set_ylabel('Number of Videos')
axes[0].legend()

top20_commented = comments_per_video.nlargest(20, 'comment_count')
axes[1].barh(top20_commented['video_id'].str[-8:], top20_commented['comment_count'], color='teal')
axes[1].set_title('Top 20 Most Commented Videos')
axes[1].set_xlabel('Comment Count')
axes[1].set_ylabel('Video ID (last 8 chars)')
axes[1].invert_yaxis()

plt.suptitle('Comment Distribution Across Videos', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f'Videos with ≥10 comments:  {(cpv >= 10).sum()}')
print(f'Videos with ≥50 comments:  {(cpv >= 50).sum()}')
print(f'Median comments per video:  {cpv.median():.1f}')

**What this tells us:**
- Most videos get very few comments — a classic **long tail** pattern.
- A few videos drive most of the conversation.
- If most videos have 1–2 comments, there's limited text signal per video — comment *count* matters more than comment *content* for recommendation.

---
## Part 3 — Authors / Creators

These plots answer: *Who are the creators in our dataset? Are they big influencers or regular users?*

### Plot 11 — Author Videos Count

How many videos has each creator posted? This tells us if we're looking at active creators or one-off posters.

In [ ]:
vc        = authors['videos_count'].dropna()
vc_clipped = vc[vc < vc.quantile(0.95)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(vc_clipped, bins=40, color='seagreen', edgecolor='white')
axes[0].axvline(vc.median(), color='red', linestyle='--', label=f'Median: {vc.median():.0f}')
axes[0].set_title('Author Video Count (top 5% removed)')
axes[0].set_xlabel('Total Videos Posted')
axes[0].set_ylabel('Number of Authors')
axes[0].legend()

top_creators = authors.nlargest(15, 'videos_count')[['username', 'videos_count']].reset_index(drop=True)
axes[1].barh(top_creators['username'].fillna('unknown'), top_creators['videos_count'], color='seagreen')
axes[1].set_title('Top 15 Most Prolific Creators')
axes[1].set_xlabel('Total Videos Posted')
axes[1].invert_yaxis()

plt.suptitle('Creator Activity in Our Dataset', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
print(vc.describe().round(1))

**What this tells us:**
- If most authors have very few videos, our dataset skews towards casual creators rather than power users.
- The right chart names the most prolific creators — the ones who post the most content.
- Creator identity is a strong recommendation signal — users often follow creators they like, not just topics.

### Plot 12 — Avg Video Plays per Author

How big is each creator's typical audience? This separates micro-creators from viral ones.

In [ ]:
ap        = authors['avg_video_plays'].dropna()
ap_clipped = ap[ap < ap.quantile(0.95)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(ap_clipped, bins=40, color='darkorange', edgecolor='white')
axes[0].axvline(ap.median(), color='red', linestyle='--', label=f'Median: {ap.median():,.0f}')
axes[0].set_title('Avg Plays per Author (top 5% removed)')
axes[0].set_xlabel('Avg Plays per Video')
axes[0].set_ylabel('Number of Authors')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
axes[0].legend()

top_plays = authors.nlargest(15, 'avg_video_plays')[['username', 'avg_video_plays', 'videos_count']].reset_index(drop=True)
axes[1].barh(top_plays['username'].fillna('unknown'), top_plays['avg_video_plays'], color='darkorange')
axes[1].set_title('Top 15 Creators by Avg Plays')
axes[1].set_xlabel('Avg Plays per Video')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
axes[1].invert_yaxis()

plt.suptitle('Creator Reach in Our Dataset', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**What this tells us:**
- If heavily skewed, a few viral creators dominate reach in our dataset.
- A creator with 500k avg plays but only 3 videos is more signal-rich than someone with 200 videos averaging 1k plays.
- High-reach creators are useful anchors for content-based recommendation.

---
## Part 4 — Relationships Between Variables

These plots answer: *Which signals actually predict engagement? What correlates with what?*

### Plot 13 — Correlation Heatmap

Shows how much each pair of metrics moves together. Values close to +1 = strong positive link. Close to -1 = opposite. Close to 0 = no relationship.

In [ ]:
corr_cols = ['likes', 'plays', 'shares', 'comments_count', 'engagement_rate', 'duration_sec']
corr      = videos[corr_cols].dropna().corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, ax=ax, annot_kws={'size': 11})
ax.set_title('Video Metrics Correlation Matrix\n(+1 = perfect positive, -1 = perfect negative)', pad=15)
plt.tight_layout()
plt.show()

**How to read this:**
- The diagonal is always 1.0 (a metric always correlates with itself — ignore it).
- **Red** = positive (both go up together). **Blue** = negative (one up, other down).
- Key things to look for:
  - `likes` vs `plays` — if high, popular videos just get more of everything
  - `engagement_rate` vs `plays` — if near 0 or negative, high-view videos aren't more engaging
  - `duration_sec` vs anything — tells you if length actually matters
- A good recommender uses features that don't just track raw popularity — we want to surface good content that hasn't gone viral yet.

### Plot 14 — Plays vs Likes (colored by Engagement Rate)

Are high-play videos always high-engagement? Log scale lets us see the full range — from 10 views to millions — without squishing the data.

In [ ]:
pl = videos[['plays', 'likes', 'engagement_rate']].dropna()
pl = pl[(pl['plays'] > 0) & (pl['likes'] >= 0)]

fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(
    pl['plays'], pl['likes'],
    c=pl['engagement_rate'], cmap='plasma',
    alpha=0.4, s=15,
    norm=plt.Normalize(vmin=0, vmax=pl['engagement_rate'].quantile(0.95))
)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Engagement Rate')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Plays (log scale)')
ax.set_ylabel('Likes (log scale)')
ax.set_title('Plays vs Likes — colored by Engagement Rate')
plt.tight_layout()
plt.show()

**How to read this:**
- Each dot is one video. **Color** = engagement rate. Bright yellow/orange = high engagement.
- If bright dots are scattered everywhere (not just top-right), **engagement rate is independent of raw popularity** — small videos can be just as engaging as viral ones.
- This is the core insight for our project: **popularity ≠ quality**. A good recommender uses engagement rate, not just plays.

---
## Summary — Key Takeaways

| Finding | Implication for Recommendation |
|---|---|
| Engagement rate is right-skewed | A small % of videos are much better content — don't treat all equally |
| Popularity ≠ engagement rate | Use `engagement_rate` as quality signal, not just play count |
| Most comments are very short | Comment *count* matters more than comment *text* for signal |
| Comment activity is concentrated | Long tail problem — most videos have almost no comments |
| Duration varies widely | Use duration as a user preference feature |
| Hashtags cluster around a few topics | Useful for content-based filtering by topic |
| Most creators post few videos | Creator identity is sparse — some accounts have very little history |

These findings should directly inform which features go into the recommendation model.